In [7]:
# ================================================================
# Model Performance Panel – ROC / PR / Lift / Confusion Matrix
# For unified lead-scoring pipeline (Bank Lead Scoring Project)
# ================================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)

# -----------------------
# 1. CONFIGURATION
# -----------------------

# Files (relative to notebooks/ directory)
SCORES_PATH = Path("../data/processed/unified_lead_scores.csv")
LABELS_PATH = Path("../data/processed/modeling_dataset_leads_unified.csv")

# Join key and columns
KEY_COL   = "lead_id"          # present in both files
LABEL_COL = "label_conv"       # CHANGE to 'label_conv_90d' if you want 90-day conversion
PROBA_COL = "ml_unified_proba" # from unified_lead_scores.csv
PRED_COL  = None               # or 'predicted_label' if you ever store hard labels

THRESHOLD = 0.5                # probability cutoff for 0/1 classification

REPORT_DIR = Path("../reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Scores file   :", SCORES_PATH.resolve())
print("Labels file   :", LABELS_PATH.resolve())
print("Saving outputs:", REPORT_DIR.resolve())

# -----------------------
# 2. LOAD DATA & MERGE
# -----------------------

if not SCORES_PATH.exists():
    raise FileNotFoundError(f"Scores file not found: {SCORES_PATH}")

if not LABELS_PATH.exists():
    raise FileNotFoundError(f"Labels file not found: {LABELS_PATH}")

df_scores = pd.read_csv(SCORES_PATH)
df_labels = pd.read_csv(LABELS_PATH)

print("\nScores columns:", list(df_scores.columns))
print("Labels columns:", list(df_labels.columns))

# Basic column checks
for col in (KEY_COL, PROBA_COL):
    if col not in df_scores.columns:
        raise ValueError(f"Column '{col}' not found in {SCORES_PATH.name}")

for col in (KEY_COL, LABEL_COL):
    if col not in df_labels.columns:
        raise ValueError(f"Column '{col}' not found in {LABELS_PATH.name}")

# Optional: deduplicate by lead_id just in case
df_scores = df_scores.drop_duplicates(subset=[KEY_COL])
df_labels = df_labels.drop_duplicates(subset=[KEY_COL])

# Merge on lead_id
df_merged = pd.merge(
    df_labels[[KEY_COL, LABEL_COL]],
    df_scores[[KEY_COL, PROBA_COL]],
    on=KEY_COL,
    how="inner",
)

print(f"\nMerged rows: {len(df_merged)}")

# Drop missing values and extract arrays
df_eval = df_merged[[LABEL_COL, PROBA_COL]].dropna()
y_true  = df_eval[LABEL_COL].astype(int).values
y_score = df_eval[PROBA_COL].astype(float).values

print("Eval rows     :", len(df_eval))
print("Positive rate :", y_true.mean().round(4))

# -----------------------
# 3. DERIVE PREDICTIONS
# -----------------------
if PRED_COL is not None and PRED_COL in df_merged.columns:
    y_pred = df_merged.loc[df_eval.index, PRED_COL].astype(int).values
else:
    y_pred = (y_score >= THRESHOLD).astype(int)
    print(f"Predictions derived using threshold = {THRESHOLD}")

# Common plot settings
plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["figure.dpi"] = 120

# -----------------------
# 4. ROC CURVE
# -----------------------
fpr, tpr, _ = roc_curve(y_true, y_score)
auc = roc_auc_score(y_true, y_score)

plt.figure()
plt.plot(fpr, tpr, label=f"ROC (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve – Unified Champion Model")
plt.legend()
plt.tight_layout()
roc_path = REPORT_DIR / "roc_curve_unified.png"
plt.savefig(roc_path, bbox_inches="tight")
plt.close()
print("Saved ROC curve to:", roc_path)

# -----------------------
# 5. PRECISION–RECALL CURVE
# -----------------------
precision, recall, _ = precision_recall_curve(y_true, y_score)
ap = average_precision_score(y_true, y_score)

plt.figure()
plt.plot(recall, precision, label=f"PR (AP = {ap:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curve – Unified Champion Model")
plt.legend()
plt.tight_layout()
pr_path = REPORT_DIR / "pr_curve_unified.png"
plt.savefig(pr_path, bbox_inches="tight")
plt.close()
print("Saved PR curve to:", pr_path)

# -----------------------
# 6. CONFUSION MATRIX + CLASSIFICATION REPORT
# -----------------------
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=[0, 1])

plt.figure()
disp.plot(values_format="d", cmap="Blues")
plt.title(f"Confusion Matrix (threshold = {THRESHOLD:.2f})")
plt.tight_layout()
cm_path = REPORT_DIR / "confusion_matrix_unified.png"
plt.savefig(cm_path, bbox_inches="tight")
plt.close()
print("Saved confusion matrix to:", cm_path)

report = classification_report(y_true, y_pred, output_dict=False)
txt_path = REPORT_DIR / "classification_report_unified.txt"
with open(txt_path, "w") as f:
    f.write(report)
print("Saved classification report to:", txt_path)
print("\nClassification report:\n")
print(report)

# -----------------------
# 7. LIFT CHART (DECILES)
# -----------------------
df_lift = (
    pd.DataFrame({"y_true": y_true, "y_score": y_score})
    .sort_values("y_score", ascending=False)
    .reset_index(drop=True)
)

df_lift["decile"] = pd.qcut(df_lift.index, 10, labels=False) + 1  # 1..10

lift_table = df_lift.groupby("decile").agg(
    leads=("y_true", "size"),
    responders=("y_true", "sum"),
)
lift_table["response_rate"] = lift_table["responders"] / lift_table["leads"]

baseline = df_lift["y_true"].mean()
lift_table["lift"] = lift_table["response_rate"] / baseline

lift_table_path = REPORT_DIR / "lift_table_unified.csv"
lift_table.to_csv(lift_table_path, index=True)
print("Saved lift table to:", lift_table_path)

plt.figure()
plt.plot(lift_table.index, lift_table["lift"], marker="o")
plt.axhline(1.0, color="gray", linestyle="--", label="Baseline")
plt.xlabel("Decile (1 = highest score)")
plt.ylabel("Lift vs baseline")
plt.title("Lift Chart – Unified Champion Model")
plt.xticks(lift_table.index)
plt.legend()
plt.tight_layout()
lift_plot_path = REPORT_DIR / "lift_chart_unified.png"
plt.savefig(lift_plot_path, bbox_inches="tight")
plt.close()
print("Saved lift chart to:", lift_plot_path)

print("\nDone. Figures ready to insert into your report:")
print(" -", roc_path.name)
print(" -", pr_path.name)
print(" -", cm_path.name)
print(" -", lift_plot_path.name)
print("Plus:")
print(" -", txt_path.name)
print(" -", lift_table_path.name)



Scores file   : /Users/mayanksuryavanshi/bank_lead_scoring_project/data/processed/unified_lead_scores.csv
Labels file   : /Users/mayanksuryavanshi/bank_lead_scoring_project/data/processed/modeling_dataset_leads_unified.csv
Saving outputs: /Users/mayanksuryavanshi/bank_lead_scoring_project/reports

Scores columns: ['cust_id', 'Customer_ID', 'lead_id', 'ml_unified_proba']
Labels columns: ['Customer_ID', 'First_Name', 'Last_Name', 'Gender', 'DOB', 'Email', 'Phone', 'City', 'State', 'Postal_Code', 'Date_Of_Lead', 'Lead_Source', 'Occupation', 'Annual_Income', 'Marital_Status', 'Customer_Segment', 'Risk_Score', 'Best_First_Option', 'Best_Second_Option', 'Chosen_Product', 'chosen_product_id', 'best_first_product_id', 'lead_id', 'cust_id', 'lead_product_name_norm', 'product_id', 'target_product_id', 'source_id', 'label_conv', 'label_conv_90d', 'lead_date', 'cif_id', 'full_name', 'gender', 'dob', 'age', 'aadhaar_no', 'pan_no', 'income_annual_inr', 'income_segment', 'occupation', 'credit_score',

<Figure size 720x480 with 0 Axes>